# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LeylaAghayeva1/ml-search-engineering/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

### 1 My baseline rule

My baseline identifies pages that are good candidates for a content refresh.

The rule prioritizes content that:
- has not been updated for a long time,
- still has meaningful search demand,
- and continues to receive impressions.

The assumption is that refreshing older pages with existing visibility provides a better opportunity than updating pages with little or no demand.

### Reason Codes

| Reason Code | Meaning |
|-------------|---------|
| STALE_HIGH_DEMAND | Old content with strong search demand |
| STALE_VISIBLE | Old content still receiving impressions |
| HIGH_PRIORITY_REFRESH | Old content with both demand and visibility |

### Verdict

#### Signal 1: days_since_last_update

**Verdict: MIXED**

The relationship between content staleness and impressions is not consistent across all buckets. Pages that have not been updated for a moderate amount of time tend to have the highest average impressions, while the oldest pages have substantially lower impressions and search volume. This suggests that content age alone is not enough to determine refresh priority and should be combined with additional signals such as search demand and visibility.

---

#### Signal 2: search_volume

**Verdict: MIXED**

Higher search volume does not consistently correspond to higher average impressions in this dataset. While pages with larger search volume remain valuable candidates, the average click-through rate decreases as search volume increases. This indicates that search volume alone is not sufficient for prioritization and should be used together with other signals.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
from pathlib import Path

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")

print(df.shape)
df.head()
# -----------------------
# Signal 1
# -----------------------

signal1 = (
    df.assign(
        bucket=pd.qcut(
            df["days_since_last_update"],
            5,
            duplicates="drop"
        )
    )
    .groupby("bucket")
    .agg(
        n=("content_id","count"),
        avg_impressions=("impressions_90d","mean"),
        avg_search_volume=("search_volume","mean")
    )
)

print("Signal 1: days_since_last_update")
print(signal1)


# -----------------------
# Signal 2
# -----------------------

signal2 = (
    df.assign(
        bucket=pd.qcut(
            df["search_volume"],
            5,
            duplicates="drop"
        )
    )
    .groupby("bucket")
    .agg(
        n=("content_id","count"),
        avg_impressions=("impressions_90d","mean"),
        avg_ctr=("ctr","mean")
    )
)

print("\nSignal 2: search_volume")
print(signal2)

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*
## Building the ranked queue

The baseline score combines three transparent signals:

- days_since_last_update ≥ 180
- search_volume ≥ 100
- impressions_90d ≥ 500

Each page receives a simple score based on these conditions. Pages satisfying more conditions receive higher scores. The ranked queue includes a reason code and an action label so that each recommendation can be explained without using a machine learning model.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import numpy as np
import pandas as pd
from pathlib import Path

# ----------------------------
# Rule conditions
# ----------------------------

stale = df["days_since_last_update"] >= 180
high_demand = df["search_volume"] >= 100
visible = df["impressions_90d"] >= 500

# ----------------------------
# Transparent baseline score
# ----------------------------

df["baseline_score"] = (
    stale.astype(int) * 3 +
    high_demand.astype(int) * 2 +
    visible.astype(int)
)

df["baseline_score"] = (
    df["baseline_score"]
    + np.log1p(df["search_volume"])
)
# ----------------------------
# Reason Codes
# ----------------------------

df["reason_code"] = np.select(
    [
        stale & high_demand & visible,
        stale & high_demand,
        stale & visible
    ],
    [
        "HIGH_PRIORITY_REFRESH",
        "STALE_HIGH_DEMAND",
        "STALE_VISIBLE"
    ],
    default="LOW_PRIORITY"
)

df["action"] = np.where(
    df["baseline_score"] > 0,
    "REFRESH_CONTENT",
    "NO_ACTION"
)

queue = (
    df.sort_values("baseline_score", ascending=False)
      .reset_index(drop=True)
)

Path("work/outputs").mkdir(parents=True, exist_ok=True)

queue.to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)

print(queue[
    ["content_id",
     "baseline_score",
     "reason_code",
     "action"]
].head(10))

## 3. Top-20 review
### Top-20 review

The highest-ranked pages are expected to be older pages that still attract search demand and impressions. These are reasonable refresh candidates because improvements may increase existing traffic.

Confidence is highest when all three signals (staleness, demand, and visibility) are present.

A recommendation could be incorrect if:
- search demand has permanently declined,
- the page is already scheduled for an update,
- seasonal effects temporarily reduce performance,
- impressions are driven by irrelevant keywords.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
top20 = queue[
    [
        "content_id",
        "baseline_score",
        "reason_code",
        "action",
        "days_since_last_update",
        "search_volume",
        "impressions_90d"
    ]
].head(20)

for i, row in top20.iterrows():
    print(f"{i+1}.")
    print(f"Action: {row.action}")
    print(f"Reason: {row.reason_code}")
    print("Confidence: Medium-High")
    print("Could be wrong if the page is seasonal or already planned for refresh.\n")

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*
### Weak picks and leakage check

Some lower-ranked pages may still receive high scores because they satisfy the rule despite having other limitations such as low click-through rate or outdated search demand.

The baseline intentionally remains simple and transparent instead of trying to capture every edge case.

Leakage check:
- trend_direction was NOT used.
- trend_pct was NOT used.
- IDs were NOT used as features.
- No future-window information was included.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("Leakage Check")

print("trend_direction used:",
      "trend_direction" in ["days_since_last_update","search_volume","impressions_90d"])

print("trend_pct used:",
      "trend_pct" in ["days_since_last_update","search_volume","impressions_90d"])

print("No label-derived features used.")

weak = queue.tail(10)

print("\nExample weak picks")
print(weak[
    [
        "content_id",
        "baseline_score",
        "reason_code"
    ]
])

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.